# SOXX 2019 年两份原始正文身份校验

只扫描两份已 NAS 回读的原始正文，并要求报告日期、SOXX 投资明细标题、Security/Shares/Value/Common Stocks 表头相连。

In [1]:
import json,hashlib,re,html
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_2019_transition_bodies_20261023';plan=json.loads((card/'plan.json').read_text());transport=json.loads((card/'transport_result.json').read_text());receipts=json.loads((card/'capture_receipts.json').read_text())
assert transport['status']=='two_exact_2019_old_form_bodies_archived_identity_pending' and transport['nas_remote_and_scoped_restore_verified']
raw=root/'data/snapshots/stage1_soxx_2019_transition_bodies_20261023';rows=[]
for spec in plan['exact_bodies']:
 receipt=next(x for x in receipts if x['accession']==spec['accession']);p=raw/(spec['accession']+'.html');b=p.read_bytes();assert hashlib.sha256(b).hexdigest()==receipt['raw_sha256']
 target='December 31, 2018' if spec['report_period']=='2018-12-31' else 'March 31, 2019';matches=[]
 for m in re.finditer(b'PHLX Semiconductor ETF',b,re.I):
  start=max(0,m.start()-20000);end=min(len(b),m.start()+20000);z=b[start:end].decode('latin1');z=re.sub(r'<[^>]*>',' ',z);z=re.sub(r'\s+',' ',html.unescape(z));anchor=re.search(r'Schedule of Investments(?: \(unaudited\))? '+re.escape(target)+r' iShares [^ ]* PHLX Semiconductor ETF \(Percentages shown are based on Net Assets\) Security Shares Value Common Stocks',z,re.I)
  if anchor and b'<TABLE' in b[m.start():end]:matches.append({'title_offset':m.start(),'header_excerpt':z[anchor.start():anchor.end()+95],'has_table_markup':True})
 rows.append({'accession':spec['accession'],'report_period':spec['report_period'],'raw_sha256':receipt['raw_sha256'],'raw_bytes':len(b),'matching_dated_SOXX_schedule_headers':len(matches),'matching_headers':matches,'identity_passed':len(matches)>=1})
result={'status':'two_2019_transition_original_SOXX_schedule_identities_passed' if len(rows)==2 and all(x['identity_passed'] for x in rows) else 'body_identity_stop','source_plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'NAS_archive_sha256':transport['nas_archive_sha256'],'NAS_scoped_restore_verified':True,'rows':rows,'numeric_holdings_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'identity_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({'status':result['status'],'matches':[(x['accession'],x['matching_dated_SOXX_schedule_headers']) for x in rows]})

{'status': 'two_2019_transition_original_SOXX_schedule_identities_passed', 'matches': [('0001193125-19-059323', 1), ('0001193125-19-167652', 1)]}


## 中文结论

本卡仅证明两份原始正文存在 SOXX 自身的当期投资明细标题；尚未完成持仓数值和 Stage1 特征。